# 06. Continuous training: new data arrives, the model retrains while the endpoint keeps serving

Design: `docs/retraining.md`. Code: `pipelines/retrain_pipeline.py`, `pipelines/triggers.py`, `bankpulse_core/ingest.py`.

1. **Triggers (BP-1105):** a new file in `raw/incoming/`, a monthly schedule, or the drift alarm starts the pipeline.
2. **Pipeline (BP-1104):** Prepare → Train → Evaluate (challenger vs champion) → Check → Register (PendingManualApproval).
3. **Arrivals (BP-1107):** the history is replayed month by month (26 calendar months, May 2008 – Nov 2010).
4. **Results:** each execution's data window, metrics and decision, and the registry's version history.

Nothing here changes or deletes the deployed endpoint. New versions wait for a person to approve them.

In [ ]:
# Setup: repository root, config, lab connection and the state saved by earlier notebooks.
from pathlib import Path

import pandas as pd
import sagemaker

from bankpulse_core.aws import lab_boto_session, read_state, s3_uri, update_state
from bankpulse_core.config import load_config

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
config = load_config(ROOT / 'config' / 'data_prep.yaml')
STATE = Path(config['output']['processed_dir']) / 'sagemaker_state.json'
state = read_state(STATE)

boto_session = lab_boto_session()                       # bankpulse-lab profile; no keys in code
sm_session = sagemaker.Session(boto_session=boto_session)
sm_client = boto_session.client('sagemaker')
bucket, prefix, role = state['bucket'], state['prefix'], state['role']

import json

from bankpulse_core.ingest import read_manifest
from bankpulse_core.io import read_json

PIPELINE = 'bankpulse-retrain'
events = boto_session.client('events')
batches = s3_uri(bucket, prefix, 'raw', 'batches')

## 1. Triggers

Three EventBridge rules start the same pipeline, as `LabRole`:
- `bankpulse-retrain-on-upload`: an S3 *Object Created* event under `raw/incoming/`. The bucket sends its events to EventBridge.
- `bankpulse-retrain-monthly`: 03:00 UTC on the 1st of each month.
- `bankpulse-retrain-on-drift`: the feature-drift alarm from notebook 05 enters `ALARM`.

In [ ]:
rules = [r for r in events.list_rules(NamePrefix='bankpulse-retrain')['Rules']]
display(pd.DataFrame([{'rule': r['Name'], 'state': r['State'],
                       'trigger': r.get('ScheduleExpression') or json.loads(r['EventPattern'])['detail-type'][0],
                       'targets': [t['Arn'].rsplit('/', 1)[1] for t in events.list_targets_by_rule(Rule=r['Name'])['Targets']]}
                      for r in rules]))

## 2. Data batches: what has arrived

Every accepted monthly file is listed in `raw/batches/manifest.json` with its row count, subscription rate and SHA-256. Files that break the data contract go to `raw/rejected/` with a report, and are never used.

In [ ]:
manifest = pd.DataFrame(read_manifest(batches)['batches'])
display(manifest[['id', 'rows', 'positive_rate', 'added_at']].tail(8))
print(len(manifest), 'batches accepted;', int(manifest['rows'].sum()), 'rows in total')
ax = manifest.set_index('id')['positive_rate'].plot(figsize=(11, 3), marker='o', title='Subscription rate by monthly batch')
ax.set_xticks(range(len(manifest))); ax.set_xticklabels(manifest['id'].str.slice(0, 4), rotation=90, fontsize=7);

## 3. Pipeline executions

Each execution:
1. ingests whatever is waiting in `raw/incoming/`;
2. trains on the **latest 12 months**, validates on the next 2, and holds out the **newest 2**;
3. scores **both** the new model (challenger) and the deployed one (champion) on that holdout, each encoded with its own feature schema;
4. registers the challenger only if it passes the rule in `docs/retraining.md`.

In [ ]:
def execution_summary(execution):
    arn = execution['PipelineExecutionArn']
    steps = {s['StepName']: s for s in sm_client.list_pipeline_execution_steps(PipelineExecutionArn=arn)['PipelineExecutionSteps']}
    row = {'execution': arn.rsplit('/', 1)[1], 'status': execution['PipelineExecutionStatus'],
           'started': str(execution['StartTime'])[:19],
           'steps': ' > '.join(f"{name}:{steps[name]['StepStatus'][:4]}" for name in
                               ('Prepare', 'Train', 'Evaluate', 'CheckChallenger', 'Register-RegisterModel') if name in steps)}
    params = {p['Name']: p['Value'] for p in sm_client.list_pipeline_parameters_for_execution(
        PipelineExecutionArn=arn)['PipelineParameters']}
    row['capped_campaign'] = params.get('CappedCampaignVersion', 'v1')
    if steps.get('Evaluate', {}).get('StepStatus') == 'Succeeded':
        job = steps['Evaluate']['Metadata']['ProcessingJob']['Arn'].split('/')[-1]
        uri = sm_client.describe_processing_job(ProcessingJobName=job)['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
        evaluation = read_json(uri + '/evaluation.json')
        row.update({'holdout_rows': evaluation['holdout_rows'],
                    'champion_pr_auc': evaluation['champion']['pr_auc'],
                    'challenger_pr_auc': evaluation['challenger']['pr_auc'],
                    'challenger_lift': evaluation['challenger']['lift_top_decile'],
                    'challenger_age_ratio': evaluation['challenger']['subgroup_min_max_ratio'],
                    'registered': bool(evaluation['challenger_ok'])})
    if steps.get('Prepare', {}).get('StepStatus') == 'Succeeded':
        job = steps['Prepare']['Metadata']['ProcessingJob']['Arn'].split('/')[-1]
        uri = sm_client.describe_processing_job(ProcessingJobName=job)['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
        report = read_json(uri + '/ingest_report.json')
        row['ingested'] = ', '.join(f"{i['source'].rsplit('/', 1)[-1]} ({i['status']})" for i in report['ingested']) or '-'
    return row


executions = sm_client.list_pipeline_executions(PipelineName=PIPELINE, SortBy='CreationTime',
                                                SortOrder='Ascending')['PipelineExecutionSummaries']
runs = pd.DataFrame([execution_summary(e) for e in executions])
display(runs)

## 4. Champion vs challenger

On every holdout, the retrained challenger beats the 2008-trained champion by a wide margin. A model that learns from the latest 12 months follows the change in subscription behaviour (6% → 50%); the frozen one can't.

In [ ]:
import matplotlib.pyplot as plt

scored = runs.dropna(subset=['challenger_pr_auc'])
ax = scored.set_index('execution')[['champion_pr_auc', 'challenger_pr_auc']].plot.bar(
    figsize=(9, 4), rot=0, color=['#E07B00', '#2F6DB5'], title='Holdout PR-AUC: deployed champion vs retrained challenger')
ax.set_ylim(0, 1); ax.set_ylabel('PR-AUC on the newest 2 months')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8)
plt.tight_layout()

## 5. Model Registry version history

Version 1 is the model trained in notebook 02, which is approved and deployed. Each successful pipeline execution adds a version as **PendingManualApproval**, with links to its own feature schema and evaluation. Approving one and switching the endpoint (`pipelines/deploy.py`, a blue/green canary with auto-rollback) is a deliberate, separate step.

In [ ]:
versions = []
for summary in sm_client.list_model_packages(ModelPackageGroupName=state['model_package_group'], SortBy='CreationTime',
                                             SortOrder='Ascending')['ModelPackageSummaryList']:
    detail = sm_client.describe_model_package(ModelPackageName=summary['ModelPackageArn'])
    versions.append({'version': detail['ModelPackageVersion'], 'status': detail['ModelApprovalStatus'],
                     'created': str(detail['CreationTime'])[:19],
                     'source': 'pipeline' if 'evaluation_uri' in detail.get('CustomerMetadataProperties', {}) else 'notebook 02'})
display(pd.DataFrame(versions))
endpoint = sm_client.describe_endpoint(EndpointName=state['endpoint'])
print('endpoint', state['endpoint'], 'is', endpoint['EndpointStatus'], '- still serving version 1 (unchanged)')